In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "insomnia_dataset" for domain "fitbit_activity" and was generated for All of Us Controlled Tier Dataset v8
dataset_48616985_fitbit_activity_sql <- paste("
    SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary   
    WHERE
        activity_summary.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (436962, 42538607, 37110488, 4102985, 439013)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_activity_48616985_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_activity_48616985",
  "fitbit_activity_48616985_*.csv")
message(str_glue('The data will be written to {fitbit_activity_48616985_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_48616985_fitbit_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_activity_48616985_path,
  destination_format = "CSV")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260420/fitbit_activity_48616985/fitbit_activity_48616985_*.csv. Use this path when reading the data into your notebooks in the future.



In [2]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_activity_48616985_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_activity <- read_bq_export_from_workspace_bucket(fitbit_activity_48616985_path)

dim(insomnia_activity)

head(insomnia_activity, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260420/fitbit_activity_48616985/fitbit_activity_48616985_000000000000.csv.



[1] 3728199      13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
9739387,2020-06-02,359,1388,1670,NA,0,NA,117,NA,606,2494,0
9739387,2020-06-29,399,1388,1709,NA,0,NA,125,NA,759,2522,0
9739387,2020-06-05,446,1387,1826,NA,8,NA,75,NA,810,5204,23
9739387,2020-08-13,78,1389,1458,NA,0,NA,26,NA,936,421,0
9739387,2021-03-14,691,1389,1972,NA,19,NA,97,NA,1294,7036,30


In [4]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')

In [3]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Insomnia/', df = insomnia_activity)


Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


corrplot 0.95 loaded



── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  98.0   1st Qu.: 106.0  
 Median : 348.0   Median : 573.5  
 Mean   : 733.3   Mean   :1048.6  
 3rd Qu.:1133.0   3rd Qu.:1820.0  
 Max.   :5386.0   Max.   :5386.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000181   Min.   :   1.0   Min.   :2009-01-02   Min.   :2015-09-20  
 1st Qu.:1811614   1st Qu.:  98.0   1st Qu.:2017-08-17   1st Qu.:2023-07-25  
 Median :2949307   Median : 348.0   Median :2021-09-26   Median :2023-10-01  
 Mean   :3951130   Mean   : 733.3   Mean   :2020-06-08   Mean   :2023-04-21  
 3rd Qu.:5946627   3rd Qu.:1133.0   3rd Qu.:2023-06-13   3rd Qu.:2023-10-01  
 Max.   :9993734   Max.   :5386.0   Max.   :2023-09-29   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing              compliance_tier
 Min.   :   1.0   Min.   :   0.0   Min.   : 0.000   High (>90%)      :2894  
 1st Qu.: 106.0   1st Qu.:   0.0   1st Qu.: 0.000   Moderate (70-90%): 876  
 Median : 573.5   Median :  22.0   Median : 6.103   Low (50-70%)     : 511  
 Mean   :1048.6   Mean   : 315.3   Mean   :19.687   Poor (<50%)      : 803  
 3rd Qu.:1820.0   3rd Qu.: 308.2   3rd Qu.:31.480                    

NULL

In [5]:
patient_stats <- activity_prep_stats(path.out = '/home/jupyter/workspaces/Data/', cohort = 'Insomnia', df = insomnia_activity)


# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            5084 2197.   550.  2.13e3 1.80e3 2503.  1.44e+3 3.21e3
2 fairly_active_minutes   5084   13.6   13.5 9.83e0 4.41e0   18.5 6.80e-1 3.93e1
3 lightly_active_minutes  5084  174.    74.5 1.74e2 1.25e2  221.  4.96e+1 2.98e2
4 sedentary_minutes       5084  916.   205.  8.83e2 7.65e2 1057.  6.33e+2 1.30e3
5 steps                   5084 5710.  3273.  5.39e3 3.35e3 7604.  1.04e+3 1.15e4
6 very_active_minutes     5084   12.5   15.3 7.12e0 2.52e0   17.1 2.68e-1 4.16e1
